# Enclave Inference — Gemma 3 — Model Owner

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `ENCLAVE_EMAIL` | Trusted execution environment |
| **Model owner** | (this notebook) | Owns the Gemma 3 model (weights + inference engine) |
| **Benchmark owner** | (separate notebook) | Owns AI safety evaluation prompts |
| **Researcher** | `RESEARCHER_EMAIL` | Submits inference job for bias/safety evaluation |

This notebook drives only the Model Owner steps; the Benchmark Owner and Researcher run their own notebooks in parallel.

## Setup

| Size | Parameters | RAM needed | Notes |
|------|-----------|------------|-------|
| 270m | 270M | ~1 GB | Fast, good for testing |
| 1b | 1B | ~3 GB | |
| 4b | 4B | ~10 GB | |
| 12b | 12B | ~29 GB | |
| 27b | 27B | ~65 GB | Requires large-memory machine |

In [ ]:
!uv pip install "jax[cpu]" flax orbax-checkpoint sentencepiece kagglehub

In [ ]:
import csv
import json
import os
import random
import shutil
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds
from gemma_inference import MODEL_CONFIGS

# ─── Choose model size here ─────────────────────────────────────────────────
MODEL_SIZE = "270m"  # Options: "270m", "1b", "4b", "12b", "27b"
# ───────────────────────────────────────────────────────────────────────────

MODEL_CFG = MODEL_CONFIGS[MODEL_SIZE]
KAGGLE_HANDLE = MODEL_CFG["kaggle_handle"]
CKPT_SUBDIR = MODEL_CFG["ckpt_subdir"]

print(f"Model size   : {MODEL_SIZE}")
print(f"Kaggle handle: {KAGGLE_HANDLE}")
print(f"Checkpoint   : {CKPT_SUBDIR}")

In [ ]:
ENCLAVE_EMAIL    = "test.enclave@gmail.com"
RESEARCHER_EMAIL = "test.researcher@gmail.com"

print(f"  Enclave    : {ENCLAVE_EMAIL}\n  Researcher : {RESEARCHER_EMAIL}")

## Step 0 — Log in as Model Owner

In [ ]:
model_owner = login_do()
print(f"  Model owner : {model_owner.email}")

In [ ]:
# Optionally to clear state
# model_owner._manager.delete_syftbox()
# model_owner._manager.peer_manager.write_own_version()

## Step 1 — Peer with the Enclave

In [ ]:
model_owner.add_peer(ENCLAVE_EMAIL)
model_owner.sync()
print(f"  Model owner peered with enclave ({ENCLAVE_EMAIL})")

### Step 1.1 — Wait for the Researcher peer request, then approve

The Researcher notebook adds you as a peer. Re-run the cell below until you see their request appear, then approve.

In [ ]:
model_owner.peers

In [ ]:
model_owner.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
model_owner.sync()
print("  Researcher peer approved")

## Step 2 — Download Gemma 3 weights from Kaggle

Weights are cached after first run.

In [ ]:
import subprocess
cmd = f'uv run --with kagglehub python -c "import kagglehub; print(kagglehub.model_download(\'{KAGGLE_HANDLE}\', force_download=False))"'
print(f"Downloading: {KAGGLE_HANDLE}")
subprocess.run(cmd, shell=True, check=True)

In [ ]:
# Resolve weights from kagglehub cache (works with either download method)
weights_dir = os.path.expanduser(
    f"~/.cache/kagglehub/models/google/gemma-3/flax/{CKPT_SUBDIR}/1"
)
assert os.path.isdir(weights_dir), f"Weights not found at {weights_dir} — run the download cell first"

print(f"Weights directory: {weights_dir}")
print(f"Contents: {os.listdir(weights_dir)}")

## Step 3 — Build the private dataset

Model owner's private contribution is a directory containing:
- `gemma_inference.py` — the inference engine (model architecture + generate function)
- `{CKPT_SUBDIR}/` — the checkpoint weights directory
- `tokenizer.model` — the SentencePiece tokenizer

The **mock** (public) side is just a model card describing the model.

In [ ]:
# Build Model owner's private dataset directory: inference code + weights + tokenizer
INFERENCE_MODULE = Path("gemma_inference.py").resolve()
assert INFERENCE_MODULE.exists(), f"Missing {INFERENCE_MODULE}"


def create_model_private_dir() -> Path:
    """Bundle inference code + weights into a single directory."""
    tmp = Path(tempfile.mkdtemp()) / f"gemma3-private-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)

    # Copy inference module
    shutil.copy2(INFERENCE_MODULE, tmp / "gemma_inference.py")

    # Copy tokenizer
    shutil.copy2(Path(weights_dir) / "tokenizer.model", tmp / "tokenizer.model")

    # Copy checkpoint directory
    ckpt_src = Path(weights_dir) / CKPT_SUBDIR
    shutil.copytree(ckpt_src, tmp / CKPT_SUBDIR)

    return tmp


def create_model_mock_file() -> Path:
    """Public model card — visible to the researcher."""
    tmp = Path(tempfile.mkdtemp()) / f"model-mock-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "model_card.txt"
    p.write_text("\n".join([
        f"Gemma 3 {MODEL_SIZE.upper()}-IT: A {MODEL_SIZE} parameter instruction-tuned language model.",
        f"{'=' * (len(MODEL_SIZE) + 12)}",
        "License: Gemma Terms of Use",
        "Intended use: Research and evaluation purposes",
        "",
        "Usage:",
        "  import gemma_inference as gemma",
        f'  model, tokenizer, params = gemma.setup("{MODEL_SIZE}", weights_dir)',
        '  response, stats = gemma.generate(model, params, tokenizer, "Your prompt here")',
        "",
    ]))
    return p

In [ ]:
model_private_dir = create_model_private_dir()
model_mock = create_model_mock_file()

print(f"Private dir contents:")
for item in sorted(model_private_dir.rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  {item.relative_to(model_private_dir)}  ({size_mb:.1f} MB)")

## Step 4 — Upload gemma3_model

Mock = model card; private = weights + inference code; shared with the enclave so it can run inference.

In [ ]:
model_owner.create_dataset(
    name="gemma3_model",
    mock_path=model_mock,
    private_path=model_private_dir,
    summary=f"Gemma 3 {MODEL_SIZE.upper()}-IT — instruction-tuned language model for safety evaluation",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    upload_private=True,
    sync=False,
)
model_owner.share_private_dataset("gemma3_model", ENCLAVE_EMAIL)
model_owner.sync()
print("  Model owner uploaded 'gemma3_model'")

## Step 5 — Wait for the Researcher to submit the job, then approve

The Researcher submits `safety_eval_job` to the enclave. Re-sync until it appears here, inspect it, then approve.

In [ ]:
JOB_NAME = "safety_eval_job"
model_owner.sync()
model_owner_job = next(j for j in model_owner.jobs if j.name == JOB_NAME)
print(f"  Model owner sees '{JOB_NAME}'  status={model_owner_job.status}")

In [ ]:
model_owner.approve_job(model_owner_job)
print("  Model owner approved")

## Step 6 — Receive results

Results arrive here because the Researcher submitted with `share_results_with_do=True`.

In [ ]:
model_owner.sync()
model_owner_job = next(j for j in model_owner.jobs if j.name == JOB_NAME)
print(f"  Output files : {model_owner_job.output_paths}")
assert len(model_owner_job.output_paths) > 0